## Previsão de Rotatividade de Clientes
### Projeto Didático — Data Science & Machine Learning Clássico

---

**Objetivo:** construir um pipeline completo de Data Science para prever quais clientes de uma empresa de telecomunicações têm maior probabilidade de cancelar o serviço (*churn*), permitindo ações de retenção proativas.

**Dataset:** [Telco Customer Churn](https://www.kaggle.com/blastchar/telco-customer-churn) (IBM Sample Dataset) — 7.043 clientes, 21 atributos.

**O que este notebook cobre:**
1. Contextualização do problema de negócio
2. Análise Exploratória de Dados (EDA)
3. Limpeza e pré-processamento
4. Engenharia de features
5. Tratamento de desbalanceamento de classes
6. Modelagem: Regressão Logística → Random Forest → XGBoost
7. Avaliação com métricas apropriadas (não só acurácia!)
8. Interpretação de resultados e importância das variáveis
9. Conclusões e próximos passos

## 1. Contexto do Problema de Negócio

**Churn** (ou taxa de cancelamento) é uma das métricas mais críticas para empresas baseadas em assinatura (telecom, streaming, SaaS, etc.). Adquirir um novo cliente custa, em média, de 5 a 25 vezes mais do que reter um cliente existente.

**Pergunta de negócio:** *Dado o perfil e o histórico de um cliente, qual a probabilidade de ele cancelar o serviço no próximo período?*

**Por que isso importa:**
- Permite à empresa priorizar ações de retenção (descontos, contato proativo, upgrade de plano) para clientes de alto risco.
- Evita desperdiçar recursos de retenção em clientes que não iriam cancelar de qualquer forma.

**Tipo de problema:** Classificação binária supervisionada (`Churn` = Yes/No).

**Desafio central deste dataset:** as classes são desbalanceadas (~73% não-churn vs ~27% churn), então acurácia sozinha é uma métrica enganosa — um modelo "preguiçoso" que sempre prevê "não vai cancelar" já acerta 73% das vezes sem aprender nada útil.


## 2. Setup do Ambiente

Importar as bibliotecas necessárias. Se estiver rodando localmente pela primeira vez, instale as dependências:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn shap jinja2
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações visuais
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Ambiente configurado com sucesso.')

## 3. Carregamento dos Dados

O dataset "Telco Customer Churn" é público (IBM Sample Data Sets). Neste projeto vamos carregá-lo diretamente.


In [ ]:
# Caminho do arquivo local (ajuste conforme necessário)
CAMINHO_CSV = 'Telco-Customer-Churn.csv'

df = pd.read_csv(CAMINHO_CSV)

print(f'Shape do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

In [ ]:
df.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df.describe()

## 4. Limpeza dos Dados

Alguns pontos de atenção conhecidos neste dataset (bugs "clássicos" que todo cientista de dados deve saber identificar):

1. **`TotalCharges`** vem como `object` (string), não `float` — porque existem 11 registros com espaço em branco `" "` no lugar de um número (clientes com `tenure = 0`, ou seja, cadastrados mas ainda sem cobrança).
2. **`customerID`** é um identificador único, sem valor preditivo — deve ser removido antes da modelagem (mas guardado para referência).
3. Várias colunas categóricas usam `"No internet service"` / `"No phone service"` como uma categoria distinta de `"No"` — vamos decidir conscientemente se simplificamos isso.


In [ ]:
# 1. Corrigindo TotalCharges
print('Valores não numéricos em TotalCharges:', (df['TotalCharges'].str.strip() == '').sum())

# Converte para numérico, forçando erros para NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f'Valores nulos após conversão: {df["TotalCharges"].isnull().sum()}')
print(df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

In [ ]:
# Confirma a hipótese: todos os NaN são clientes com tenure == 0 (acabaram de entrar)
assert (df.loc[df['TotalCharges'].isnull(), 'tenure'] == 0).all(), 'Hipótese não confirmada!'

# Para esses clientes, faz sentido que TotalCharges seja 0 (ainda não foram cobrados)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('Nulos restantes em TotalCharges:', df['TotalCharges'].isnull().sum())

In [ ]:
# 2. Guardamos customerID à parte e removemos do dataframe de trabalho
customer_ids = df['customerID'].copy()
df = df.drop(columns=['customerID'])

# 3. Verificando duplicatas
print('Linhas duplicadas:', df.duplicated().sum())

# 4. Checagem final de nulos
df.isnull().sum()[df.isnull().sum() > 0]